# A.7. Unknown-delay identification

Search ten distinct random true delays, sampled from [100, 500], using two dyadic branches followed by fixed-width joint candidate evaluation. Each embedding uses 10 ESNs, 30 histories per model and five evolution steps.

Existing observations, searches and final detections are reused; missing data or model results are computed. Dependencies: `ESN.ipynb`, `Reconstruct.ipynb`, NumPy, pandas, SciPy and IPython.


In [ ]:
import ast
import gc
import hashlib
import json
import os
import pickle
import platform
import sys
import tempfile
import time
from pathlib import Path
os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
os.environ.setdefault('OMP_NUM_THREADS', '4')
import numpy as np
import pandas as pd
import scipy
from IPython.display import display, HTML
SETTINGS = {'n_nodes': 2, 'embedding_width': 6, 'coarse_intervals': [64, 128], 'threshold': 0.0001, 'stopping_resolution': 1, 'n_train': 10000, 'n_washout': 100, 'n_test': 2000, 'units_per_input_dimension': 100, 'n_models': 10, 'n_initial_histories': 30, 'n_evolution_steps': 5, 'n_trials': 10, 'recovery_tolerance_fraction': 0.05, 'input_scaling': 0.5, 'spectral_radius': 0.9, 'mean_connectivity': 2.0, 'ridge': 0.0001, 'j_threshold': 0.01, 'exclude_self_pairs': True, 'initial_seed': 6199, 'noise_seed': 6299, 'observation_seed': 6399, 'reservoir_seed': 6499, 'offset_seed': 6699}
DELAY_SETTINGS = {'delay_range_samples': [100, 500], 'runs_per_delay': 1, 'sampling_seed': 6799}
DATA_SETTINGS = {'cause_node': 2, 'effect_node': 1, 'coupling_weight': 1.0, 'global_coupling': 0.3, 'integration_step': 0.01, 'dynamical_noise_std': 1e-06, 'observation_noise_std': 1e-06, 'burn_in': 2000, 'heterogeneity_seed': 6099}
RETRY_FAILED = False
BASE = Path.cwd()
if not (BASE / 'A.7.ipynb').exists():
    matches = list(BASE.glob('*/A.7.ipynb'))
    if len(matches) != 1:
        raise FileNotFoundError('Run from the notebook folder or its immediate parent.')
    BASE = matches[0].parent
NOTEBOOK_PATH = BASE / 'A.7.ipynb'
ESN_PATH = BASE / 'ESN.ipynb'
esn_notebook = json.loads(ESN_PATH.read_text(encoding='utf-8'))
ESN_SOURCE = '\n'.join((''.join(cell['source']) for cell in esn_notebook['cells'] if cell['cell_type'] == 'code'))
exec(compile(ESN_SOURCE, str(ESN_PATH), 'exec'), globals())
RECONSTRUCT_PATH = BASE / 'Reconstruct.ipynb'
reconstruct_notebook = json.loads(RECONSTRUCT_PATH.read_text(encoding='utf-8'))
RECONSTRUCT_SOURCE = '\n'.join((''.join(cell['source']) for cell in reconstruct_notebook['cells'] if cell['cell_type'] == 'code'))
exec(compile(RECONSTRUCT_SOURCE, str(RECONSTRUCT_PATH), 'exec'), globals())
for interval in SETTINGS['coarse_intervals']:
    ratio = interval / SETTINGS['stopping_resolution']
    if interval <= 0 or ratio < 1 or (not float(ratio).is_integer()):
        raise ValueError('Coarse intervals must be positive integer multiples of epsilon.')
    integer_ratio = int(ratio)
    if integer_ratio & integer_ratio - 1:
        raise ValueError('Use dyadic coarse/epsilon ratios to avoid rounding bisection.')
(delay_lower, delay_upper) = DELAY_SETTINGS['delay_range_samples']
coarse_support = (SETTINGS['embedding_width'] - 1) * max(SETTINGS['coarse_intervals'])
ERROR_NORMALIZATION_SPAN = delay_upper - delay_lower
n_delay_groups = (SETTINGS['n_trials'] + DELAY_SETTINGS['runs_per_delay'] - 1) // DELAY_SETTINGS['runs_per_delay']
delay_rng = np.random.RandomState(DELAY_SETTINGS['sampling_seed'])
GROUP_DELAYS = delay_rng.choice(np.arange(delay_lower, delay_upper + 1), size=n_delay_groups, replace=False).tolist()
EXPERIMENT_PLAN = []
for trial_index in range(SETTINGS['n_trials']):
    (group_index, repeat_index) = divmod(trial_index, DELAY_SETTINGS['runs_per_delay'])
    EXPERIMENT_PLAN.append({'run_id': f'run_{trial_index + 1:02d}', 'trial': trial_index, 'delay_group_id': f'group_{group_index + 1:02d}', 'repeat_in_group': repeat_index + 1, 'true_delay_samples': int(GROUP_DELAYS[group_index])})

def data_settings_for_trial(trial):
    return {**DATA_SETTINGS, 'delay_samples': EXPERIMENT_PLAN[trial]['true_delay_samples']}

def ensemble_seed(base_seed, trial, model_index):
    return int(np.random.SeedSequence([base_seed, trial, model_index]).generate_state(1, dtype=np.uint32)[0])
MAX_HISTORY = (SETTINGS['embedding_width'] + 1) * max(SETTINGS['coarse_intervals'])
HISTORY_LENGTH = MAX_HISTORY + 1
RAW_ORIGIN = HISTORY_LENGTH + DATA_SETTINGS['burn_in'] + MAX_HISTORY
TRAIN_STOP = SETTINGS['n_washout'] + SETTINGS['n_train']
TEST_START = TRAIN_STOP + MAX_HISTORY + 1
EMBEDDED_LENGTH = TEST_START + SETTINGS['n_test']
own_notebook = json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))

def cell_source(cell):
    return ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
computation_source = '\n'.join((cell_source(cell) for cell in own_notebook['cells'] if cell['cell_type'] == 'code' and cell.get('metadata', {}).get('stage') != 'unified_review'))
computation_source = '\n'.join((line for line in computation_source.splitlines() if not line.startswith('RETRY_FAILED =')))
generation_function = next((node for node in ast.parse(computation_source).body if isinstance(node, ast.FunctionDef) and node.name == 'generate_observations'))

def canonical_ast(node):
    if isinstance(node, ast.AST):
        return {'node': type(node).__name__, 'fields': {name: canonical_ast(value) for (name, value) in ast.iter_fields(node) if value is not None and value != []}}
    if isinstance(node, list):
        return [canonical_ast(value) for value in node]
    return node
DATA_SIGNATURE = {'format': 'further_study_6_observations_v1', 'generator_ast_sha256': hashlib.sha256(json.dumps(canonical_ast(generation_function), sort_keys=True).encode('utf-8')).hexdigest(), 'settings': {key: SETTINGS[key] for key in ('n_nodes', 'n_washout', 'n_train', 'n_test', 'initial_seed', 'noise_seed', 'observation_seed')}, 'data_settings': DATA_SETTINGS, 'delay_settings': DELAY_SETTINGS, 'experiment_plan': EXPERIMENT_PLAN, 'layout': {'max_history': MAX_HISTORY, 'history_length': HISTORY_LENGTH, 'raw_origin': RAW_ORIGIN, 'train_stop': TRAIN_STOP, 'test_start': TEST_START, 'embedded_length': EMBEDDED_LENGTH}}
DATA_SIGNATURE_TOKEN = hashlib.sha256(json.dumps(DATA_SIGNATURE, sort_keys=True).encode('utf-8')).hexdigest()
current_search_source_hash = hashlib.sha256(computation_source.encode('utf-8')).hexdigest()
search_cache_compatibility = own_notebook.get('metadata', {}).get('search_cache_compatibility', {})
search_computation_hash = current_search_source_hash
if current_search_source_hash == search_cache_compatibility.get('verified_current_source_sha256'):
    search_computation_hash = search_cache_compatibility['previous_computation_sha256']
SIGNATURE = {'pipeline': 'further_study_6_grouped_random_delays_range_normalized_error_v6', 'ddi_definition': 'signed_source_mean_normalization_positive_history_scores_ensemble_matrix_mean', 'ensemble_order': 'positive_history_mean_per_model_then_elementwise_model_mean_then_max', 'training_pairs_per_model': SETTINGS['n_train'] - 1, 'data_signature': DATA_SIGNATURE_TOKEN, 'data_manifest': DATA_SIGNATURE, 'esn_sha256': hashlib.sha256(ESN_SOURCE.encode('utf-8')).hexdigest(), 'error_definition': 'absolute_delay_error_divided_by_configured_true_delay_range_width', 'error_normalization_bounds_samples': [delay_lower, delay_upper], 'error_normalization_span_samples': ERROR_NORMALIZATION_SPAN, 'exact_recovery_definition': 'normalized_delay_error_at_most_configured_tolerance_fraction', 'recovery_absolute_tolerance_samples': SETTINGS['recovery_tolerance_fraction'] * ERROR_NORMALIZATION_SPAN, 'search_grid': 'unperturbed_dyadic_refinement', 'reconstruct_sha256': hashlib.sha256(RECONSTRUCT_SOURCE.encode('utf-8')).hexdigest(), 'settings': SETTINGS, 'data_settings': DATA_SETTINGS, 'delay_settings': DELAY_SETTINGS, 'experiment_plan': EXPERIMENT_PLAN, 'computation_sha256': search_computation_hash, 'python': platform.python_version(), 'numpy': np.__version__, 'scipy': scipy.__version__, 'pandas': pd.__version__}
SIGNATURE_TOKEN = hashlib.sha256(json.dumps(SIGNATURE, sort_keys=True).encode('utf-8')).hexdigest()
RESULT_DIR = BASE / 'result' / 'A.7'
DATA_CACHE_DIR = RESULT_DIR / 'data' / DATA_SIGNATURE_TOKEN[:20]
CACHE_DIR = RESULT_DIR / 'runs' / SIGNATURE_TOKEN[:20]
TABLE_DIR = RESULT_DIR / 'tables' / SIGNATURE_TOKEN[:20]
for directory in (DATA_CACHE_DIR, CACHE_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def atomic_pickle(value, path):
    path = Path(path)
    (descriptor, temporary_path) = tempfile.mkstemp(dir=path.parent, suffix='.tmp')
    with os.fdopen(descriptor, 'wb') as stream:
        pickle.dump(value, stream, protocol=pickle.HIGHEST_PROTOCOL)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary_path, path)

class NumpyCompatibleUnpickler(pickle.Unpickler):

    def find_class(self, module, name):
        if np.__version__.split('.')[0] == '1' and (module == 'numpy._core' or module.startswith('numpy._core.')):
            module = 'numpy.core' + module[len('numpy._core'):]
        return super().find_class(module, name)

def load_pickle(path):
    with Path(path).open('rb') as stream:
        return NumpyCompatibleUnpickler(stream).load()

def recovery_fields(estimate, true_delay, tolerance_fraction, delay_range_samples):
    if not np.isfinite(true_delay) or true_delay <= 0:
        raise ValueError('True delay must be finite and positive.')
    bounds = np.asarray(delay_range_samples, dtype=float)
    if bounds.shape != (2,) or not np.all(np.isfinite(bounds)) or bounds[1] <= bounds[0]:
        raise ValueError('Error normalization requires two finite increasing delay bounds.')
    if not np.isfinite(tolerance_fraction) or not 0 <= tolerance_fraction < 1:
        raise ValueError('Use a fractional recovery tolerance in [0, 1).')
    normalization_span = float(bounds[1] - bounds[0])
    detected = bool(np.isfinite(estimate))
    error = abs(estimate - true_delay) if detected else np.nan
    return {'absolute_error_samples': error, 'relative_error': error / normalization_span, 'exact_recovery': bool(detected and error <= tolerance_fraction * normalization_span)}


## Part 1. Load observations

Load and verify the ten recorded trajectories. The Lorenz network has a delayed edge from node 2 to node 1, global coupling 0.3 and integration/sampling step 0.01. Data use training-calibrated median/IQR normalization. Missing observations stop execution unless generation is explicitly enabled.


In [ ]:
def generate_observations(data_settings, settings, trial):
    n_nodes = settings['n_nodes']
    dimension = 3 * n_nodes
    delay = int(data_settings['delay_samples'])
    if delay < 1 or delay > MAX_HISTORY:
        raise ValueError('Generation delay must fit the preallocated history.')
    adjacency = np.zeros((n_nodes, n_nodes), dtype=float)
    adjacency[data_settings['effect_node'] - 1, data_settings['cause_node'] - 1] = data_settings['coupling_weight']
    parameter_rng = np.random.RandomState(data_settings['heterogeneity_seed'])
    heterogeneity = parameter_rng.uniform(-0.06, 0.06, size=n_nodes)
    initial_rng = np.random.RandomState(settings['initial_seed'] + trial)
    noise_rng = np.random.RandomState(settings['noise_seed'] + trial)
    raw = np.empty((RAW_ORIGIN + EMBEDDED_LENGTH, dimension), dtype=float)
    raw[:HISTORY_LENGTH] = initial_rng.uniform(0.0, 1.0, size=dimension)

    def derivative(current, delayed, noise):
        state = current.reshape(n_nodes, 3)
        delayed_y = delayed[1::3]
        coupling = adjacency @ delayed_y - adjacency.sum(axis=1) * state[:, 1]
        result = np.empty_like(state)
        result[:, 0] = -10.0 * (state[:, 0] - state[:, 1] + data_settings['global_coupling'] * coupling)
        result[:, 1] = 28.0 * (1.0 + heterogeneity) * state[:, 0] - state[:, 1] - state[:, 0] * state[:, 2]
        result[:, 2] = -8.0 / 3.0 * state[:, 2] + state[:, 0] * state[:, 1]
        return result.reshape(-1) + noise
    integration_step = data_settings['integration_step']
    for index in range(HISTORY_LENGTH - 1, raw.shape[0] - 1):
        delayed_start = raw[index - delay]
        delayed_end = raw[index - delay + 1]
        delayed_middle = (delayed_start + delayed_end) / 2.0
        noise = noise_rng.normal(0.0, data_settings['dynamical_noise_std'], size=(4, dimension))
        first = derivative(raw[index], delayed_start, noise[0])
        second = derivative(raw[index] + integration_step * first / 2.0, delayed_middle, noise[1])
        third = derivative(raw[index] + integration_step * second / 2.0, delayed_middle, noise[2])
        fourth = derivative(raw[index] + integration_step * third, delayed_end, noise[3])
        raw[index + 1] = raw[index] + integration_step * (first + 2.0 * second + 2.0 * third + fourth) / 6.0
        if not np.all(np.isfinite(raw[index + 1])):
            raise FloatingPointError(f'Non-finite integration state at index {index + 1}.')
    calibration_start = RAW_ORIGIN + settings['n_washout']
    calibration_stop = RAW_ORIGIN + TRAIN_STOP
    quartiles = np.percentile(raw[calibration_start:calibration_stop], [25, 50, 75], axis=0, method='midpoint')
    median = quartiles[1]
    scale = quartiles[2] - quartiles[0]
    if np.any(scale <= 1e-12):
        raise ValueError('A training variable has a zero interquartile range.')
    observations = (raw - median) / scale
    observation_rng = np.random.RandomState(settings['observation_seed'] + trial)
    observations += observation_rng.normal(0.0, data_settings['observation_noise_std'], size=observations.shape)
    return {'data_signature': DATA_SIGNATURE_TOKEN, 'trial': trial, 'delay_group_id': EXPERIMENT_PLAN[trial]['delay_group_id'], 'repeat_in_group': EXPERIMENT_PLAN[trial]['repeat_in_group'], 'true_delay_samples': delay, 'data_settings': dict(data_settings), 'observations': observations, 'training_median': median, 'training_iqr': scale, 'heterogeneity': heterogeneity, 'calibration_raw_bounds': (calibration_start, calibration_stop), 'observations_sha256': hashlib.sha256(observations.tobytes()).hexdigest()}
(DATA_CACHE_DIR / 'manifest.json').write_text(json.dumps({'data_signature': DATA_SIGNATURE_TOKEN, **DATA_SIGNATURE}, indent=2, ensure_ascii=False), encoding='utf-8')
DATA_PATHS = []
data_cache_counts = {'loaded': 0, 'generated': 0}
for trial_index in range(SETTINGS['n_trials']):
    path = DATA_CACHE_DIR / f'observations_{trial_index + 1:02d}.pkl'
    if path.exists():
        bundle = load_pickle(path)
        if bundle.get('data_signature') != DATA_SIGNATURE_TOKEN or bundle.get('trial') != trial_index or bundle.get('true_delay_samples') != EXPERIMENT_PLAN[trial_index]['true_delay_samples'] or (bundle.get('data_settings') != data_settings_for_trial(trial_index)) or (bundle['observations'].shape != (RAW_ORIGIN + EMBEDDED_LENGTH, 3 * SETTINGS['n_nodes'])):
            raise ValueError('Dataset checkpoint signature mismatch.')
        digest = hashlib.sha256(bundle['observations'].tobytes()).hexdigest()
        if digest != bundle['observations_sha256']:
            raise ValueError('Dataset checksum mismatch.')
        data_cache_status = 'loaded from cache'
        data_cache_counts['loaded'] += 1
    else:
        bundle = generate_observations(data_settings_for_trial(trial_index), SETTINGS, trial_index)
        atomic_pickle(bundle, path)
        data_cache_status = 'generated and saved'
        data_cache_counts['generated'] += 1
    DATA_PATHS.append(path)
    del bundle
if TEST_START - MAX_HISTORY <= TRAIN_STOP:
    raise AssertionError('Delayed test inputs overlap training targets.')
(CACHE_DIR / 'manifest.json').write_text(json.dumps(SIGNATURE, indent=2, ensure_ascii=False), encoding='utf-8')
experiment_plan = pd.DataFrame(EXPERIMENT_PLAN)
experiment_plan['true_delay_physical_time'] = experiment_plan['true_delay_samples'] * DATA_SETTINGS['integration_step']
experiment_plan.to_csv(TABLE_DIR / 'A.7_experiment_plan.csv', index=False, encoding='utf-8-sig')
display(experiment_plan)


## Part 2. Define DDI and branch search

Use coarse spacings 64 and 128, threshold $10^{-4}$ and unit stopping resolution. DDI uses the existing signed source-mean denominator, near-zero rejection and positive-history mean. Average model matrices before maximizing over positive delays and distinct node pairs. Branch searches use unperturbed dyadic grids.


In [ ]:
def embed_observations(observations, positive_delays):
    delays = [0] + [int(value) for value in positive_delays]
    if delays != sorted(set(delays)) or max(delays) > MAX_HISTORY:
        raise ValueError('Invalid positive candidate delays or insufficient history allocation.')
    return np.concatenate([observations[RAW_ORIGIN - delay:RAW_ORIGIN - delay + EMBEDDED_LENGTH] for delay in delays], axis=1)

def compute_ddi(model, embedded, positive_delays, settings, offsets):
    test_data = embedded[TEST_START:TEST_START + settings['n_test']]
    matrices = np.empty((len(positive_delays), settings['n_nodes'], settings['n_nodes']))
    for (lag_index, delay) in enumerate(positive_delays):
        matrices[lag_index] = cal_time_TDI_copy_detect(model, test_data, settings['n_nodes'], 3, 3 * settings['n_nodes'], settings['n_initial_histories'], len(positive_delays) + 1, 0, 1, settings['n_washout'], settings['n_evolution_steps'], offsets, lag_index + 1, j_threshold=settings['j_threshold'])
    if not np.all(np.isfinite(matrices)):
        raise FloatingPointError('The DDI function returned a non-finite value.')
    return matrices

def evaluate_delays(observations, positive_delays, settings, trial, *, cache_dir=None, signature_token=None):
    cache_directory = CACHE_DIR if cache_dir is None else Path(cache_dir)
    checkpoint_signature = SIGNATURE_TOKEN if signature_token is None else signature_token
    positive_delays = tuple((int(value) for value in positive_delays))
    key = hashlib.sha256(json.dumps(positive_delays).encode('utf-8')).hexdigest()[:20]
    prefix = f'trial_{trial + 1:02d}_embedding_{key}'
    path = cache_directory / f'{prefix}.pkl'
    started = time.perf_counter()
    if path.exists():
        payload = load_pickle(path)
        if payload['signature'] != checkpoint_signature or tuple(payload['delays']) != positive_delays or payload['trial'] != trial:
            raise ValueError('Embedding checkpoint mismatch.')
        return {**payload, 'cache_hit': True, 'call_seconds': time.perf_counter() - started}
    embedded = embed_observations(observations, positive_delays)
    dimension = embedded.shape[1]
    n_units = settings['units_per_input_dimension'] * dimension
    member_results = []
    for model_index in range(settings['n_models']):
        member_path = cache_directory / f'{prefix}_model_{model_index + 1:02d}.pkl'
        reservoir_seed = ensemble_seed(settings['reservoir_seed'], trial, model_index)
        offset_seed = ensemble_seed(settings['offset_seed'], trial, model_index)
        offsets = np.random.RandomState(offset_seed).randint(settings['n_washout'], settings['n_test'] - settings['n_evolution_steps'] - 1, size=settings['n_initial_histories'])
        if member_path.exists():
            member = load_pickle(member_path)
            if member['signature'] != checkpoint_signature or tuple(member['delays']) != positive_delays or member['trial'] != trial or (member['model_index'] != model_index) or (member['reservoir_seed'] != reservoir_seed) or (not np.array_equal(member['test_offsets'], offsets)):
                raise ValueError('Individual-ESN checkpoint mismatch.')
        else:
            model = None
            random_state = np.random.get_state()
            try:
                np.random.seed(reservoir_seed)
                training_start = time.perf_counter()
                model = ESN(n_units, dimension, dimension)
                model.train(embedded[:settings['n_washout']], embedded[settings['n_washout']:TRAIN_STOP - 1], embedded[settings['n_washout'] + 1:TRAIN_STOP])
                training_seconds = time.perf_counter() - training_start
                scoring_start = time.perf_counter()
                matrices = compute_ddi(model, embedded, positive_delays, settings, offsets)
                scoring_seconds = time.perf_counter() - scoring_start
                member = {'signature': checkpoint_signature, 'trial': trial, 'delays': positive_delays, 'model_index': model_index, 'reservoir_seed': reservoir_seed, 'offset_seed': offset_seed, 'test_offsets': offsets, 'ddi_matrices': matrices, 'input_dimension': dimension, 'n_units': n_units, 'training_seconds': training_seconds, 'scoring_seconds': scoring_seconds}
                atomic_pickle(member, member_path)
            finally:
                np.random.set_state(random_state)
                del model
                gc.collect()
        member_results.append(member)
    member_matrices = np.stack([member['ddi_matrices'] for member in member_results])
    matrices = np.mean(member_matrices, axis=0)
    if not np.all(np.isfinite(matrices)):
        raise FloatingPointError('Non-finite ensemble-mean DDI.')
    model_rows = []
    for member in member_results:
        for (lag_index, delay) in enumerate(positive_delays):
            for effect in range(settings['n_nodes']):
                for cause in range(settings['n_nodes']):
                    if cause != effect:
                        model_rows.append({'model_index': member['model_index'] + 1, 'lag_samples': int(delay), 'cause_node': cause + 1, 'effect_node': effect + 1, 'ddi': float(member['ddi_matrices'][lag_index, effect, cause]), 'n_initial_histories': settings['n_initial_histories'], 'reservoir_seed': member['reservoir_seed'], 'offset_seed': member['offset_seed']})
    payload = {'signature': checkpoint_signature, 'trial': trial, 'delays': positive_delays, 'ddi_matrices': matrices, 'member_ddi_matrices': member_matrices, 'model_ddi_std': np.std(member_matrices, axis=0, ddof=1) if settings['n_models'] > 1 else np.full_like(matrices, np.nan), 'model_rows': model_rows, 'n_models': settings['n_models'], 'n_initial_histories_per_model': settings['n_initial_histories'], 'total_history_count': settings['n_models'] * settings['n_initial_histories'], 'training_seconds': sum((member['training_seconds'] for member in member_results)), 'scoring_seconds': sum((member['scoring_seconds'] for member in member_results)), 'input_dimension': dimension, 'n_units': n_units}
    atomic_pickle(payload, path)
    del embedded
    gc.collect()
    return {**payload, 'cache_hit': False, 'call_seconds': time.perf_counter() - started}

def peak_ddi(payload):
    matrices = np.asarray(payload['ddi_matrices']).copy()
    diagonal = np.arange(matrices.shape[1])
    matrices[:, diagonal, diagonal] = np.nan
    flat_index = int(np.nanargmax(matrices))
    (lag_index, effect, cause) = np.unravel_index(flat_index, matrices.shape)
    return {'delay': int(payload['delays'][lag_index]), 'score': float(matrices[lag_index, effect, cause]), 'effect': int(effect + 1), 'cause': int(cause + 1), 'lag_index': int(lag_index)}

def search_alg_1(evaluator, settings, run_id):
    rounds = []
    score_rows = []
    model_rows = []
    candidates = []
    width = settings['embedding_width']
    threshold = settings['threshold']
    epsilon = settings['stopping_resolution']
    focused_offsets = np.arange(width - 1) - (width - 2) // 2

    def record(branch, coarse_interval, phase, iteration, before_resolution, attempted_resolution, before_center, payload, accepted, after_center, after_resolution, accepted_strength, stop_reason):
        peak = peak_ddi(payload)
        round_id = len(rounds) + 1
        row = {'run_id': run_id, 'round_id': round_id, 'branch': branch, 'coarse_interval_samples': coarse_interval, 'phase': phase, 'iteration_in_branch': iteration, 'resolution_before_samples': before_resolution, 'temporal_resolution_samples': attempted_resolution, 'center_before_samples': before_center, 'candidate_delays_samples': json.dumps(list(payload['delays'])), 'peak_ddi_delay_samples': peak['delay'], 'maximum_ddi': peak['score'], 'peak_cause_node': peak['cause'], 'peak_effect_node': peak['effect'], 'threshold': threshold, 'accepted': bool(accepted), 'center_after_samples': after_center, 'accepted_resolution_samples': after_resolution, 'accepted_strength': accepted_strength, 'stop_reason': stop_reason, 'n_models': payload.get('n_models', np.nan), 'n_initial_histories_per_model': payload.get('n_initial_histories_per_model', np.nan), 'peak_model_ddi_std': float(payload['model_ddi_std'][peak['lag_index'], peak['effect'] - 1, peak['cause'] - 1]), 'input_dimension': payload['input_dimension'], 'n_units': payload['n_units'], 'training_seconds': payload.get('training_seconds', np.nan), 'scoring_seconds': payload.get('scoring_seconds', np.nan), 'cache_hit': payload.get('cache_hit', False)}
        rounds.append(row)
        for (lag_index, delay) in enumerate(payload['delays']):
            for effect in range(settings['n_nodes']):
                for cause in range(settings['n_nodes']):
                    if cause == effect:
                        continue
                    score_rows.append({'run_id': run_id, 'round_id': round_id, 'branch': branch, 'phase': phase, 'lag_samples': int(delay), 'cause_node': cause + 1, 'effect_node': effect + 1, 'ddi': float(payload['ddi_matrices'][lag_index, effect, cause]), 'model_ddi_std': float(payload['model_ddi_std'][lag_index, effect, cause]), 'n_models': payload['n_models'], 'n_initial_histories_per_model': payload['n_initial_histories_per_model'], 'total_history_count': payload.get('total_history_count', np.nan)})
        for sample in payload.get('model_rows', []):
            model_rows.append({'run_id': run_id, 'round_id': round_id, 'branch': branch, **sample})
        return peak

    def execute_candidate(delays, branch, coarse_interval, phase, iteration, resolution, attempted_resolution, center):
        return evaluator(delays)
    for (branch, coarse_interval) in enumerate(settings['coarse_intervals'], start=1):
        delays = tuple((coarse_interval * index for index in range(1, width)))
        payload = execute_candidate(delays, branch, coarse_interval, 'coarse', 0, coarse_interval, coarse_interval, None)
        peak = peak_ddi(payload)
        if peak['score'] < threshold:
            record(branch, coarse_interval, 'coarse', 0, coarse_interval, coarse_interval, None, payload, False, None, None, None, 'coarse_below_threshold')
            continue
        center = peak['delay']
        strength = peak['score']
        resolution = coarse_interval
        final_payload = payload
        final_peak = peak
        reason = 'resolution_reached' if resolution <= epsilon else 'continue'
        record(branch, coarse_interval, 'coarse', 0, coarse_interval, coarse_interval, None, payload, True, center, resolution, strength, reason)
        iteration = 0
        while resolution > epsilon:
            attempted_resolution = resolution / 2
            if not float(attempted_resolution).is_integer():
                raise ValueError('Noninteger bisection would require a separately specified interpolation rule.')
            attempted_resolution = int(attempted_resolution)
            focused = tuple((int(center + offset * attempted_resolution) for offset in focused_offsets if center + offset * attempted_resolution > 0))
            payload = execute_candidate(focused, branch, coarse_interval, 'focus', iteration + 1, resolution, attempted_resolution, center)
            peak = peak_ddi(payload)
            before_center = center
            before_resolution = resolution
            iteration += 1
            if peak['score'] >= threshold:
                center = peak['delay']
                strength = peak['score']
                resolution = attempted_resolution
                final_payload = payload
                final_peak = peak
                reason = 'resolution_reached' if resolution <= epsilon else 'continue'
                record(branch, coarse_interval, 'focus', iteration, before_resolution, attempted_resolution, before_center, payload, True, center, resolution, strength, reason)
            else:
                reason = 'refinement_below_threshold'
                record(branch, coarse_interval, 'focus', iteration, before_resolution, attempted_resolution, before_center, payload, False, center, resolution, strength, reason)
                break
        candidates.append({'branch': branch, 'coarse_interval_samples': coarse_interval, 'estimated_delay_samples': center, 'accepted_strength': strength, 'accepted_resolution_samples': resolution, 'stop_reason': reason, 'cause_node_at_peak': final_peak['cause'], 'effect_node_at_peak': final_peak['effect'], 'topology_ddi': final_payload['ddi_matrices'][final_peak['lag_index']].copy()})
    winner = max(candidates, key=lambda candidate: candidate['accepted_strength']) if candidates else None
    return {'rounds': rounds, 'candidate_scores': score_rows, 'model_scores': model_rows, 'candidates': candidates, 'winner': winner, 'status': 'detected' if winner is not None else 'no_detection'}


## Part 3. Search, combine candidates and report

Resume both branches, retain their distinct candidate centers and fill five positive-delay slots with seeded neighbors (default radius 2). Include the current-time block to preserve 36 inputs and 3,600 reservoir units. All candidates share one newly fitted or cached 10-ESN ensemble, with the same member seeds and test histories as the branch searches. No true labels enter candidate construction.

Final tables use the joint detection. Error is $|\widehat\tau-\tau|/(500-100)$; the 5% tolerance is 20 sampling steps. Joint recovery also requires the correct direction. All ten runs remain in the denominator. Branch records are saved separately for inspection.


In [ ]:
REVIEW_SETTINGS = {'neighborhood_radius_samples': 2, 'neighborhood_seed': 6899}
for (review_key, minimum) in (('neighborhood_radius_samples', 1), ('neighborhood_seed', 0)):
    review_value = REVIEW_SETTINGS[review_key]
    if isinstance(review_value, bool) or not isinstance(review_value, (int, np.integer)) or review_value < minimum:
        raise ValueError(f'{review_key} must be an integer >= {minimum}.')
review_implementation_source = '\n'.join((cell_source(cell) for cell in own_notebook['cells'] if cell.get('metadata', {}).get('unified_review_role') == 'implementation'))
review_source_hash = hashlib.sha256(review_implementation_source.encode('utf-8')).hexdigest()
review_compatibility = own_notebook.get('metadata', {}).get('review_cache_compatibility', {})
review_computation_hash = review_compatibility.get(review_source_hash, review_source_hash)
REVIEW_SIGNATURE = {'format': 'further_study_6_unified_review_v1', 'search_signature': SIGNATURE_TOKEN, 'review_settings': REVIEW_SETTINGS, 'implementation_sha256': review_computation_hash, 'input_blocks': SETTINGS['embedding_width'], 'selection': 'maximum_ensemble_mean_ddi_over_review_delays_and_distinct_node_pairs', 'neighborhood_rule': 'retain_all_centers_then_seeded_cyclic_unique_local_offsets', 'history_rule': 'same_reference_offsets_per_member_as_original_search', 'earlier_search_retraining': False}
REVIEW_TOKEN = hashlib.sha256(json.dumps(REVIEW_SIGNATURE, sort_keys=True).encode('utf-8')).hexdigest()
REVIEW_CACHE_DIR = RESULT_DIR / 'reviews' / REVIEW_TOKEN[:20]
REVIEW_TABLE_DIR = TABLE_DIR / 'unified_review' / REVIEW_TOKEN[:20]
for review_directory in (REVIEW_CACHE_DIR, REVIEW_TABLE_DIR):
    review_directory.mkdir(parents=True, exist_ok=True)

def construct_review_grid(candidates, width, history_limit, review_settings, trial):
    slot_count = width - 1
    centers = sorted(set((int(candidate['estimated_delay_samples']) for candidate in candidates)))
    if not centers:
        return ((), [])
    if len(centers) > slot_count:
        raise ValueError('The fixed-width review cannot retain all distinct branch centers.')
    if any((center < 1 or center > history_limit for center in centers)):
        raise ValueError('A branch candidate lies outside the allocated observation history.')
    if slot_count > history_limit:
        raise ValueError('Insufficient positive delays to fill the fixed review input dimension.')
    radius = review_settings['neighborhood_radius_samples']
    seed = ensemble_seed(review_settings['neighborhood_seed'], trial, 0)
    rng = np.random.RandomState(seed)
    selected = set(centers)
    proposals = {}
    center_order = rng.permutation(centers).tolist()
    for distance in range(1, radius + 1):
        for center in center_order:
            proposals.setdefault(center, []).extend([center - distance, center + distance])
    for center in center_order:
        rng.shuffle(proposals[center])
    while len(selected) < slot_count:
        progressed = False
        for center in center_order:
            while proposals[center]:
                proposal = int(proposals[center].pop())
                if 1 <= proposal <= history_limit and proposal not in selected:
                    selected.add(proposal)
                    progressed = True
                    break
            if len(selected) == slot_count:
                break
        if not progressed:
            radius += 1
            for center in center_order:
                proposals[center] = rng.permutation([center - radius, center + radius]).tolist()
    delays = tuple(sorted(selected))
    audit = []
    for delay in delays:
        minimum_distance = min((abs(delay - center) for center in centers))
        nearest_centers = [center for center in centers if abs(delay - center) == minimum_distance]
        audit.append({'lag_samples': delay, 'is_branch_center': delay in centers, 'nearest_center_samples': json.dumps(nearest_centers), 'nearest_branch_ids': json.dumps(sorted((int(candidate['branch']) for candidate in candidates if int(candidate['estimated_delay_samples']) in nearest_centers))), 'distance_to_nearest_center_samples': minimum_distance, 'construction_radius_samples': radius, 'neighborhood_seed': seed})
    return (delays, audit)

def run_unified_review(search_result, data_path, trial):
    if search_result['signature'] != SIGNATURE_TOKEN or search_result['trial'] != trial:
        raise ValueError('The review requires a compatible search result for this trial.')
    candidates = search_result['candidates']
    (delays, grid_audit) = construct_review_grid(candidates, SETTINGS['embedding_width'], MAX_HISTORY, REVIEW_SETTINGS, trial)
    run_id = search_result['run_id']
    if search_result['status'] == 'failed' or not delays:
        return {'run_id': run_id, 'trial': trial, 'signature': REVIEW_TOKEN, 'status': 'failed' if search_result['status'] == 'failed' else 'no_detection', 'winner': None, 'delays': delays, 'grid_audit': grid_audit, 'payload': None, 'review_cache_hit': False, 'elapsed_seconds': 0.0, 'error': search_result.get('error', '')}
    data_bundle = load_pickle(data_path)
    observations = data_bundle['observations']
    observations_digest = hashlib.sha256(observations.tobytes()).hexdigest()
    if data_bundle['data_signature'] != DATA_SIGNATURE_TOKEN or data_bundle['trial'] != trial or observations_digest != data_bundle['observations_sha256'] or (observations_digest != search_result['observations_sha256']):
        raise ValueError('Review observations do not match the original search data.')
    cache_identity = {'signature': REVIEW_TOKEN, 'trial': trial, 'delays': delays, 'observations_sha256': observations_digest, 'branch_centers': sorted(((int(candidate['branch']), int(candidate['estimated_delay_samples'])) for candidate in candidates))}
    review_path = REVIEW_CACHE_DIR / f'{run_id}_review.pkl'
    if review_path.exists():
        saved_review = load_pickle(review_path)
        if saved_review['cache_identity'] != cache_identity:
            raise ValueError('Unified-review checkpoint mismatch.')
        return {**saved_review, 'review_cache_hit': True}
    started = time.perf_counter()
    payload = evaluate_delays(observations, delays, SETTINGS, trial, cache_dir=REVIEW_CACHE_DIR, signature_token=REVIEW_TOKEN)
    expected_dimension = 3 * SETTINGS['n_nodes'] * SETTINGS['embedding_width']
    if payload['input_dimension'] != expected_dimension:
        raise AssertionError('The unified-review input dimension differs from the configured width.')
    peak = peak_ddi(payload)
    winner = peak if peak['score'] >= SETTINGS['threshold'] else None
    result = {**cache_identity, 'cache_identity': cache_identity, 'run_id': run_id, 'status': 'detected' if winner is not None else 'no_detection', 'winner': winner, 'delays': delays, 'grid_audit': grid_audit, 'payload': payload, 'review_cache_hit': False, 'elapsed_seconds': time.perf_counter() - started, 'error': ''}
    atomic_pickle(result, review_path)
    return result


In [ ]:
RUN_RESULTS = []
for (trial_index, data_path) in enumerate(DATA_PATHS):
    plan = EXPERIMENT_PLAN[trial_index]
    run_id = plan['run_id']
    result_path = CACHE_DIR / f'{run_id}_search.pkl'
    if result_path.exists():
        saved = load_pickle(result_path)
        if saved['signature'] != SIGNATURE_TOKEN or saved['true_delay_samples'] != plan['true_delay_samples']:
            raise ValueError('Search checkpoint signature mismatch.')
        if saved['status'] != 'failed' or not RETRY_FAILED:
            RUN_RESULTS.append(saved)
            continue
    started = time.perf_counter()
    observations = None
    data_bundle = None
    data_bundle = load_pickle(data_path)
    observations = data_bundle['observations']

    def evaluator(delays):
        return evaluate_delays(observations, delays, SETTINGS, trial_index)
    result = search_alg_1(evaluator, SETTINGS, run_id)
    result['observations_sha256'] = data_bundle['observations_sha256']
    result.update({'signature': SIGNATURE_TOKEN, **plan, 'initial_seed': SETTINGS['initial_seed'] + trial_index, 'noise_seed': SETTINGS['noise_seed'] + trial_index, 'observation_seed': SETTINGS['observation_seed'] + trial_index, 'n_models_per_embedding': SETTINGS['n_models'], 'n_initial_histories_per_model': SETTINGS['n_initial_histories'], 'reservoir_seeds': [ensemble_seed(SETTINGS['reservoir_seed'], trial_index, model_index) for model_index in range(SETTINGS['n_models'])], 'offset_seeds': [ensemble_seed(SETTINGS['offset_seed'], trial_index, model_index) for model_index in range(SETTINGS['n_models'])], 'elapsed_seconds': time.perf_counter() - started})
    atomic_pickle(result, result_path)
    RUN_RESULTS.append(result)
    del observations, data_bundle
    gc.collect()

def aggregate_recovery(frame, planned_runs):
    if len(frame) != planned_runs:
        raise ValueError('Every planned run must be present in the recovery denominator.')
    return {'planned_runs': planned_runs, 'recorded_runs': len(frame), 'failed_runs': int((frame['status'] == 'failed').sum()), 'no_detection_runs': int((frame['status'] == 'no_detection').sum()), 'detected_runs': int((frame['status'] == 'detected').sum()), 'mean_absolute_error_detected_runs': float(frame['absolute_error_samples'].mean()), 'mean_relative_error_detected_runs': float(frame['relative_error'].mean()), 'peak_edge_correct_count': int(frame['peak_edge_correct'].sum()), 'peak_edge_accuracy': float(frame['peak_edge_correct'].sum() / planned_runs), 'joint_delay_direction_recovery_count': int(frame['joint_delay_direction_recovery'].sum()), 'joint_delay_direction_recovery_rate': float(frame['joint_delay_direction_recovery'].sum() / planned_runs), 'exact_recovery_count': int(frame['exact_recovery'].sum()), 'exact_recovery_rate': float(frame['exact_recovery'].sum() / planned_runs), 'within_resolution_count': int(frame['within_stopping_resolution'].sum()), 'within_resolution_rate': float(frame['within_stopping_resolution'].sum() / planned_runs)}


In [ ]:
if len(RUN_RESULTS) != SETTINGS['n_trials'] or len(DATA_PATHS) != SETTINGS['n_trials']:
    raise RuntimeError('Load all planned search results and observation paths before unified review.')
if [result['run_id'] for result in RUN_RESULTS] != [plan['run_id'] for plan in EXPERIMENT_PLAN]:
    raise ValueError('Search results must follow the complete prespecified experiment plan.')
REVIEW_RESULTS = []
for (trial_index, (search_result, data_path)) in enumerate(zip(RUN_RESULTS, DATA_PATHS)):
    reviewed = run_unified_review(search_result, data_path, trial_index)
    REVIEW_RESULTS.append(reviewed)
summary_rows = []
review_grid_rows = []
review_score_rows = []
review_model_rows = []
review_topology_rows = []
for (search_result, reviewed) in zip(RUN_RESULTS, REVIEW_RESULTS):
    labels = {key: search_result[key] for key in ('run_id', 'delay_group_id', 'repeat_in_group', 'true_delay_samples')}
    winner = reviewed['winner']
    estimate = winner['delay'] if winner is not None else np.nan
    recovery = recovery_fields(estimate, search_result['true_delay_samples'], SETTINGS['recovery_tolerance_fraction'], DELAY_SETTINGS['delay_range_samples'])
    edge_correct = bool(winner is not None and winner['cause'] == DATA_SETTINGS['cause_node'] and (winner['effect'] == DATA_SETTINGS['effect_node']))
    summary_rows.append({**labels, 'status': reviewed['status'], 'branch_candidates_samples': json.dumps({str(candidate['branch']): int(candidate['estimated_delay_samples']) for candidate in search_result['candidates']}, sort_keys=True), 'candidate_delays_samples': json.dumps(list(reviewed['delays'])), 'estimated_delay_samples': estimate, **recovery, 'within_stopping_resolution': bool(winner is not None and recovery['absolute_error_samples'] <= SETTINGS['stopping_resolution']), 'peak_cause_node': winner['cause'] if winner is not None else np.nan, 'peak_effect_node': winner['effect'] if winner is not None else np.nan, 'selected_strength': winner['score'] if winner is not None else np.nan, 'peak_edge_correct': edge_correct, 'joint_delay_direction_recovery': bool(recovery['exact_recovery'] and edge_correct), 'input_dimension': reviewed['payload']['input_dimension'] if reviewed['payload'] is not None else np.nan, 'n_units': reviewed['payload']['n_units'] if reviewed['payload'] is not None else np.nan, 'n_models': SETTINGS['n_models'], 'n_initial_histories_per_model': SETTINGS['n_initial_histories'], 'cache_hit': reviewed['review_cache_hit'], 'elapsed_seconds': reviewed['elapsed_seconds'], 'error': reviewed['error']})
    review_grid_rows.extend(({**labels, **row} for row in reviewed['grid_audit']))
    payload = reviewed['payload']
    if payload is not None:
        review_model_rows.extend(({**labels, **row} for row in payload['model_rows']))
        for (lag_index, delay) in enumerate(payload['delays']):
            for effect in range(SETTINGS['n_nodes']):
                for cause in range(SETTINGS['n_nodes']):
                    if cause == effect:
                        continue
                    score_row = {**labels, 'lag_samples': int(delay), 'cause_node': cause + 1, 'effect_node': effect + 1, 'ddi': float(payload['ddi_matrices'][lag_index, effect, cause]), 'model_ddi_std': float(payload['model_ddi_std'][lag_index, effect, cause])}
                    review_score_rows.append(score_row)
                    if winner is not None and lag_index == winner['lag_index']:
                        review_topology_rows.append(score_row.copy())
summary = pd.DataFrame(summary_rows)
aggregate = pd.DataFrame([aggregate_recovery(summary, SETTINGS['n_trials'])])
group_rows = []
for (group_id, frame) in summary.groupby('delay_group_id', sort=True):
    group_rows.append({'delay_group_id': group_id, 'true_delay_samples': int(frame['true_delay_samples'].iloc[0]), **aggregate_recovery(frame, int((experiment_plan['delay_group_id'] == group_id).sum()))})
group_summary = pd.DataFrame(group_rows)
branch_iteration_rows = []
branch_candidate_rows = []
branch_score_rows = []
branch_model_rows = []
for search_result in RUN_RESULTS:
    labels = {key: search_result[key] for key in ('run_id', 'delay_group_id', 'repeat_in_group', 'true_delay_samples')}
    branch_iteration_rows.extend(({**row, **labels} for row in search_result['rounds']))
    branch_score_rows.extend(({**row, **labels} for row in search_result['candidate_scores']))
    branch_model_rows.extend(({**row, **labels} for row in search_result['model_scores']))
    branch_candidate_rows.extend(({**labels, **{key: value for (key, value) in candidate.items() if key != 'topology_ddi'}} for candidate in search_result['candidates']))
tables = {'summary': summary, 'aggregate': aggregate, 'group_summary': group_summary, 'experiment_plan': experiment_plan, 'branch_iterations': pd.DataFrame(branch_iteration_rows), 'branch_candidates': pd.DataFrame(branch_candidate_rows), 'branch_candidate_scores': pd.DataFrame(branch_score_rows), 'branch_model_scores': pd.DataFrame(branch_model_rows), 'grid_audit': pd.DataFrame(review_grid_rows), 'candidate_scores': pd.DataFrame(review_score_rows), 'model_scores': pd.DataFrame(review_model_rows), 'selected_topology': pd.DataFrame(review_topology_rows)}
for (table_name, frame) in tables.items():
    frame.to_csv(REVIEW_TABLE_DIR / f'A.7_{table_name}.csv', index=False, encoding='utf-8-sig')
review_manifest = {**REVIEW_SIGNATURE, 'detection_signature': REVIEW_TOKEN, 'error_normalization_bounds_samples': DELAY_SETTINGS['delay_range_samples'], 'error_normalization_span_samples': ERROR_NORMALIZATION_SPAN, 'recovery_tolerance_fraction': SETTINGS['recovery_tolerance_fraction'], 'planned_runs': SETTINGS['n_trials'], 'input_dimension': 3 * SETTINGS['n_nodes'] * SETTINGS['embedding_width'], 'n_models': SETTINGS['n_models'], 'n_initial_histories_per_model': SETTINGS['n_initial_histories'], 'report_source_sha256': hashlib.sha256('\n'.join((cell_source(cell) for cell in own_notebook['cells'] if cell.get('metadata', {}).get('unified_review_role') == 'report')).encode('utf-8')).hexdigest()}
(REVIEW_TABLE_DIR / 'A.7_manifest.json').write_text(json.dumps(review_manifest, indent=2, ensure_ascii=False), encoding='utf-8')
reading_columns = ['run_id', 'true_delay_samples', 'branch_candidates_samples', 'candidate_delays_samples', 'estimated_delay_samples', 'absolute_error_samples', 'relative_error', 'exact_recovery', 'peak_cause_node', 'peak_effect_node', 'peak_edge_correct', 'joint_delay_direction_recovery', 'status']
summary_view = summary[reading_columns]
html_sections = ['<!doctype html><html><head><meta charset="utf-8"><title>Unknown-delay detection results</title>', '<style>body{font-family:Arial,sans-serif;margin:28px}table{border-collapse:collapse;font-size:12px}th,td{border:1px solid #ccc;padding:7px}h2{margin-top:28px}</style></head><body>', '<h1>A.7. Final unknown-delay detection results</h1>', '<p>Each experiment uses dyadic branch searches followed by one fixed-width joint evaluation of the branch candidates and seeded neighbors. The reported estimates and recovery metrics are the final joint-evaluation results. All planned runs remain in the denominator. Branch search histories and candidate provenance are available in the audit CSV files.</p>', f"<p>Delay error is normalized by the configured range width ({ERROR_NORMALIZATION_SPAN} samples); the recovery tolerance is {SETTINGS['recovery_tolerance_fraction']:.1%} of this width. All delays are in sampling steps; joint recovery also requires the correct direction.</p>"]
for (title, frame) in [('Final detections', summary_view), ('Aggregate recovery', aggregate)]:
    html_sections.extend([f'<h2>{title}</h2>', frame.to_html(index=False, float_format=lambda value: f'{value:.7g}', escape=True)])
html_sections.append('</body></html>')
report_path = REVIEW_TABLE_DIR / 'A.7_results.html'
report_path.write_text('\n'.join(html_sections), encoding='utf-8')
atomic_pickle({'manifest': review_manifest, 'search_runs': RUN_RESULTS, 'runs': REVIEW_RESULTS, 'tables': tables}, REVIEW_CACHE_DIR / 'A.7.pkl')
display(summary_view)
display(aggregate)
print(report_path)
